In [1]:
import scipy
import os
import numpy as np
import pandas as pd
import tsfel

# ============================================================
# PARAMETERS
# ============================================================

data_path = r"C:\Users\Juliette\Research\Projects\hackathon2026\data"

FS = 1000          # Original sampling rate (Hz)
NEW_FS = 250       # Downsampled rate (Hz)
DOWNSAMPLE = FS // NEW_FS

# Windowing parameters
WINDOW_SIZE_SEC = 0.4
STEP_SEC = 0.04
PHYSIO_LAG_SEC = 0.1

# Known hardware lag
HARDWARE_LAG_SEC = 0.037

window_size = int(WINDOW_SIZE_SEC * NEW_FS)
step = int(STEP_SEC * NEW_FS)
physio_lag = int(PHYSIO_LAG_SEC * NEW_FS)
hardware_shift = int(round(HARDWARE_LAG_SEC * NEW_FS))

In [2]:
# ============================================================
# 1. LOAD SUBJECT 1
# ============================================================

sub1 = scipy.io.loadmat(os.path.join(data_path, "sub1_comp.mat"))

sub1 = {
    k: v for k, v in sub1.items()
    if k in ["train_data", "test_data", "train_dg"]
}

print("Loaded keys:", sub1.keys())

# ============================================================
# 2. DOWNSAMPLE
# ============================================================

for key in sub1:
    sub1[key] = scipy.signal.decimate(
        sub1[key],
        DOWNSAMPLE,
        axis=0,
        zero_phase=True
    )

print("Train ECoG shape after downsampling:", sub1["train_data"].shape)
print("Train glove shape after downsampling:", sub1["train_dg"].shape)

Loaded keys: dict_keys(['train_data', 'test_data', 'train_dg'])
Train ECoG shape after downsampling: (100000, 62)
Train glove shape after downsampling: (100000, 5)


In [3]:


# ============================================================
# 3. ALIGN
# ============================================================

train_ecog = sub1["train_data"][:-hardware_shift, :]
train_glove = sub1["train_dg"][hardware_shift:, :]

print("Aligned ECoG shape:", train_ecog.shape)
print("Aligned glove shape:", train_glove.shape)

# ============================================================
# 4. WINDOWING
# ============================================================

def create_windows(data, targets=None, window_size=100, step=25, lag=0):

    X = []
    y = []

    max_idx = len(data) - window_size - lag
    print("Max start index for windows:", max_idx)

    for start in range(0, max_idx, step):
        end = start + window_size

        X.append(data[start:end, :])

        if targets is not None:
            future_target = targets[
                end + lag:end + lag + step,
                :
            ].mean(axis=0)

            y.append(future_target)

    if targets is None:
        return X
    return X, np.array(y)


X_windows, y_train = create_windows(
    train_ecog,
    train_glove,
    window_size,
    step,
    physio_lag
)

print("Number of windows:", len(X_windows))
print("Target shape:", y_train.shape)

test_ecog = sub1["test_data"][:-hardware_shift, :]

X_test_windows = create_windows(
    test_ecog,
    targets=None,
    window_size=window_size,
    step=step,
    lag=physio_lag
)


Aligned ECoG shape: (99991, 62)
Aligned glove shape: (99991, 5)
Max start index for windows: 99866
Number of windows: 9987
Target shape: (9987, 5)
Max start index for windows: 49866


In [4]:

# ============================================================
# 5. TSFEL FEATURE EXTRACTION
# ============================================================

cfg = tsfel.get_features_by_domain("spectral")

feature_list = []

for i, window in enumerate(X_windows):

    df_window = pd.DataFrame(window)

    features = tsfel.time_series_features_extractor(
        cfg,
        df_window,
        fs=NEW_FS,
        verbose=0
    )

    feature_list.append(features)

    if i % 100 == 0:
        print(f"Processed window {i}/{len(X_windows)}")

X_train = pd.concat(feature_list, ignore_index=True)

print("Final feature matrix:", X_train.shape)
print("Final target matrix:", y_train.shape)

# ============================================================
# TEST FEATURES
# ============================================================

test_feature_list = []

for window in X_test_windows:

    df = pd.DataFrame(window)

    feat = tsfel.time_series_features_extractor(
        cfg, df, fs=NEW_FS, verbose=0
    )

    test_feature_list.append(feat)

X_test = pd.concat(test_feature_list, ignore_index=True)

# ============================================================
# >>>>>>>>>>>>>>>>>> ONLY ADDITION (SAVING) <<<<<<<<<<<<<<<<<<
# ============================================================

cache_path = os.path.join(data_path, "sub1_tsfel_cache.npz")

np.savez(
    cache_path,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test
)

print("Saved TSFEL features to:", cache_path)

Processed window 0/9987
Processed window 100/9987
Processed window 200/9987
Processed window 300/9987
Processed window 400/9987
Processed window 500/9987
Processed window 600/9987
Processed window 700/9987
Processed window 800/9987
Processed window 900/9987
Processed window 1000/9987
Processed window 1100/9987
Processed window 1200/9987
Processed window 1300/9987
Processed window 1400/9987
Processed window 1500/9987
Processed window 1600/9987
Processed window 1700/9987
Processed window 1800/9987
Processed window 1900/9987
Processed window 2000/9987
Processed window 2100/9987
Processed window 2200/9987
Processed window 2300/9987
Processed window 2400/9987
Processed window 2500/9987
Processed window 2600/9987
Processed window 2700/9987
Processed window 2800/9987
Processed window 2900/9987
Processed window 3000/9987
Processed window 3100/9987
Processed window 3200/9987
Processed window 3300/9987
Processed window 3400/9987
Processed window 3500/9987
Processed window 3600/9987
Processed win

In [32]:
len(feature_list)

9987

In [67]:
recovered_feature_names = feature_list[0].columns.tolist()

print(len(recovered_feature_names))   # should be 6882
print(recovered_feature_names[:20])   # inspect first few

# save as txt file for later inspection
with open(os.path.join(data_path, "tsfel_spectral_feature_names.txt"), "w") as f:
    for name in recovered_feature_names:
        f.write(name + "\n")

6882
['0_Fundamental frequency', '0_Human range energy', '0_LPCC_0', '0_LPCC_1', '0_LPCC_10', '0_LPCC_11', '0_LPCC_2', '0_LPCC_3', '0_LPCC_4', '0_LPCC_5', '0_LPCC_6', '0_LPCC_7', '0_LPCC_8', '0_LPCC_9', '0_MFCC_0', '0_MFCC_1', '0_MFCC_10', '0_MFCC_11', '0_MFCC_2', '0_MFCC_3']


In [26]:
# LOAD saved npz file
cache_path = os.path.join(data_path, "sub1_tsfel_cache.npz")
data = np.load(cache_path, allow_pickle=True)
X_train = data["X_train"]
y_train = data["y_train"]
X_test = data["X_test"]

In [ ]:
# from sklearn.feature_selection import VarianceThreshold
# from sklearn import preprocessing

# # Highly correlated features are removed
# corr_features, X_train = tsfel.correlated_features(X_train, drop_correlated=True)
# X_test.drop(corr_features, axis=1, inplace=True)

# # Remove low variance features
# selector = VarianceThreshold()
# X_train = selector.fit_transform(X_train)
# X_test = selector.transform(X_test)

# # Normalising Features
# scaler = preprocessing.StandardScaler()
# nX_train = scaler.fit_transform(X_train)
# nX_test = scaler.transform(X_test)

In [21]:
y_train.shape

(9987, 5)

In [38]:
X_train.shape, y_train.shape, X_test.shape

((9987, 6882), (9987, 5), (4987, 6882))

In [44]:
y_train.shape

(9987, 5)

In [64]:
from sklearn.feature_selection import f_regression
from sklearn.linear_model import Ridge
import numpy as np

n_fingers = y_train.shape[1]

selected_features_per_finger = []
models = []
scalers = []

y_pred_test = np.zeros((X_test.shape[0], n_fingers))

for finger in range(n_fingers):

    # ---------------------------
    # FEATURE SELECTION (per finger)
    # ---------------------------
    F, _ = f_regression(X_train, y_train[:, finger])

    top_k_idx = np.argsort(F)[-50:]

    selected_features_per_finger.append(
        np.array(recovered_feature_names)[top_k_idx]
    )

    X_train_sel = X_train[:, top_k_idx]
    X_test_sel = X_test[:, top_k_idx]

    # ---------------------------
    # SCALING (per finger, important)
    # ---------------------------
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()

    X_train_sel = scaler.fit_transform(X_train_sel)
    X_test_sel = scaler.transform(X_test_sel)

    scalers.append(scaler)

    # ---------------------------
    # MODEL (per finger)
    # ---------------------------
    model = Ridge(alpha=10.0)
    model.fit(X_train_sel, y_train[:, finger])

    models.append(model)

    y_pred_test[:, finger] = model.predict(X_test_sel)

In [65]:
# ============================================================
# EVALUATION DATA
# ============================================================

true_sig_dg = scipy.io.loadmat(os.path.join(data_path, "sub1_testlabels.mat"))
true_sig_dg = true_sig_dg["test_dg"]

test_time_idx = []

for start in range(0, len(test_ecog) - window_size - physio_lag, step):
    end = start + window_size
    test_time_idx.append(end + physio_lag)

test_time_idx = np.array(test_time_idx)

scale = 4
test_time_idx = (test_time_idx * scale).astype(int)

T = true_sig_dg.shape[0]
y_eval = np.zeros((T, 5))
counts = np.zeros((T, 5))

for i, t in enumerate(test_time_idx):
    start = max(0, t - step // 2)
    end = min(T, t + step // 2)

    y_eval[start:end] += y_pred_test[i]
    counts[start:end] += 1

counts[counts == 0] = 1
y_eval = y_eval / counts

In [66]:
from scipy.stats import pearsonr
import numpy as np

scores = []

for finger in range(5):
    r, _ = pearsonr(true_sig_dg[:, finger], y_eval[:, finger])
    scores.append(r)

print("Final score:", np.mean([scores[0], scores[1], scores[2], scores[4]])) # do not include finger 3 (ring) which is too correlated with middle finger and little finger

Final score: 0.16589419522465976


OLD

In [46]:
from sklearn.feature_selection import f_regression
import numpy as np

scores = np.zeros(X_train.shape[1])

for finger in range(5):
    F, _ = f_regression(X_train, y_train[:, finger])
    scores += F

scores /= 5  # average importance across fingers

top_k_idx = np.argsort(scores)[-200:]

X_train_sel = X_train[:, top_k_idx]
X_test_sel = X_test[:, top_k_idx]

selected_features = np.array(recovered_feature_names)[top_k_idx]

print("Number selected:", len(selected_features))

Number selected: 200


In [47]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

model = Ridge(alpha=10.0)
model.fit(X_train_scaled, y_train)

y_test_pred = model.predict(X_test_scaled)

In [49]:
# ============================================================
# EVALUATION DATA
# ============================================================

true_sig_dg = scipy.io.loadmat(os.path.join(data_path, "sub1_testlabels.mat"))
true_sig_dg = true_sig_dg["test_dg"]

test_time_idx = []

for start in range(0, len(test_ecog) - window_size - physio_lag, step):
    end = start + window_size
    test_time_idx.append(end + physio_lag)

test_time_idx = np.array(test_time_idx)

scale = 4
test_time_idx = (test_time_idx * scale).astype(int)

T = true_sig_dg.shape[0]
y_eval = np.zeros((T, 5))
counts = np.zeros((T, 5))

for i, t in enumerate(test_time_idx):
    start = max(0, t - step // 2)
    end = min(T, t + step // 2)

    y_eval[start:end] += y_test_pred[i]
    counts[start:end] += 1

counts[counts == 0] = 1
y_eval = y_eval / counts

In [50]:
from scipy.stats import pearsonr
import numpy as np

scores = []

for finger in range(5):
    r, _ = pearsonr(true_sig_dg[:, finger], y_eval[:, finger])
    scores.append(r)

print("Final score:", np.mean(scores))

Final score: 0.18162055562062088


OLD

In [43]:
from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(score_func=f_regression, k=200)
X_train_sel = selector.fit_transform(X_train, y_train[:, 0])  # or avg target
X_test_sel = selector.transform(X_test)

# Boolean mask of selected features
mask = selector.get_support()

# Names of selected features
# selected_features = X_train.columns[mask]
selected_features =  np.array(recovered_feature_names)[mask]

print("Number selected:", len(selected_features))
print(selected_features.tolist())

Number selected: 200
['0_Human range energy', '0_LPCC_5', '0_LPCC_7', '0_MFCC_5', '0_Spectral decrease', '0_Wavelet energy_12.5Hz', '0_Wavelet energy_15.62Hz', '0_Wavelet energy_20.83Hz', '0_Wavelet energy_31.25Hz', '0_Wavelet energy_62.5Hz', '0_Wavelet standard deviation_12.5Hz', '0_Wavelet standard deviation_15.62Hz', '0_Wavelet standard deviation_20.83Hz', '0_Wavelet standard deviation_31.25Hz', '0_Wavelet standard deviation_62.5Hz', '0_Wavelet variance_12.5Hz', '0_Wavelet variance_15.62Hz', '0_Wavelet variance_20.83Hz', '0_Wavelet variance_31.25Hz', '0_Wavelet variance_62.5Hz', '16_Wavelet energy_62.5Hz', '16_Wavelet standard deviation_62.5Hz', '16_Wavelet variance_62.5Hz', '20_LPCC_4', '20_LPCC_8', '20_Median frequency', '20_Spectral centroid', '20_Spectral kurtosis', '20_Spectral skewness', '20_Spectral slope', '27_Wavelet energy_20.83Hz', '27_Wavelet energy_31.25Hz', '27_Wavelet energy_62.5Hz', '27_Wavelet standard deviation_20.83Hz', '27_Wavelet standard deviation_31.25Hz', '27

In [17]:
scores = pd.DataFrame({
    "feature": X_train.columns,
    "score": selector.scores_
})

scores = scores.sort_values("score", ascending=False)

print(scores.head(20))

                                       feature       score
4030                      42_Spectral kurtosis  471.511593
4010                                 42_MFCC_0  366.020897
4034                      42_Spectral skewness  363.555540
3540      38_Wavelet standard deviation_62.5Hz  326.305321
3530                  38_Wavelet energy_62.5Hz  325.884142
4066   42_Spectrogram mean coefficient_84.68Hz  301.586436
4036                        42_Spectral spread  300.817119
3549                38_Wavelet variance_62.5Hz  299.436998
4069   42_Spectrogram mean coefficient_96.77Hz  282.549384
4045  42_Spectrogram mean coefficient_120.97Hz  273.983398
4043  42_Spectrogram mean coefficient_116.94Hz  260.579848
4067   42_Spectrogram mean coefficient_88.71Hz  246.447927
4025                        42_Power bandwidth  238.565086
4046   42_Spectrogram mean coefficient_125.0Hz  235.147034
4042   42_Spectrogram mean coefficient_112.9Hz  224.754117
4039  42_Spectrogram mean coefficient_100.81Hz  219.7086

In [15]:
X_train_sel.shape

(9987, 200)

In [5]:
# ============================================================
# MODELING
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

min_len = min(X_train_pca.shape[0], len(y_train))

X_train_pca = X_train_pca[:min_len]
y_train = y_train[:min_len]

model = Ridge(alpha=10.0)
model.fit(X_train_pca, y_train)

y_test_pred = model.predict(X_test_pca)



In [6]:
# ============================================================
# EVALUATION DATA
# ============================================================

true_sig_dg = scipy.io.loadmat(os.path.join(data_path, "sub1_testlabels.mat"))
true_sig_dg = true_sig_dg["test_dg"]

test_time_idx = []

for start in range(0, len(test_ecog) - window_size - physio_lag, step):
    end = start + window_size
    test_time_idx.append(end + physio_lag)

test_time_idx = np.array(test_time_idx)

scale = 4
test_time_idx = (test_time_idx * scale).astype(int)

T = true_sig_dg.shape[0]
y_eval = np.zeros((T, 5))
counts = np.zeros((T, 5))

for i, t in enumerate(test_time_idx):
    start = max(0, t - step // 2)
    end = min(T, t + step // 2)

    y_eval[start:end] += y_test_pred[i]
    counts[start:end] += 1

counts[counts == 0] = 1
y_eval = y_eval / counts

from scipy.stats import pearsonr
import numpy as np

scores = []

for finger in range(5):

    if finger == 3:
        continue

    r, _ = pearsonr(true_sig_dg[:, finger], y_eval[:, finger])
    scores.append(r)

print("Final score:", np.mean(scores))

Final score: 0.10353517820217975


In [8]:
import matplotlib.pyplot as plt

In [10]:
%matplotlib qt
plt.plot(y_test_pred[:, 0])

In [11]:
X_train.shape, y_train.shape, X_test.shape

((9987, 6882), (9987, 5), (4987, 6882))

In [12]:
X_train

,0_Fundamental frequency,0_Human range energy,0_LPCC_0,0_LPCC_1,0_LPCC_10,0_LPCC_11,0_LPCC_2,0_LPCC_3,0_LPCC_4,0_LPCC_5,...,9_Wavelet standard deviation_8.93Hz,9_Wavelet variance_10.42Hz,9_Wavelet variance_12.5Hz,9_Wavelet variance_15.62Hz,9_Wavelet variance_20.83Hz,9_Wavelet variance_31.25Hz,9_Wavelet variance_6.94Hz,9_Wavelet variance_62.5Hz,9_Wavelet variance_7.81Hz,9_Wavelet variance_8.93Hz
0,5.0,0.101948,0.017077,1.962646,0.030599,1.962646,0.030599,0.154558,0.317362,0.020461,...,5763.324300,2.829650e+07,2.198132e+07,1.463194e+07,7.705848e+06,3.146983e+06,3.588336e+07,516536.377942,3.589961e+07,3.321591e+07
1,5.0,0.113015,0.014735,1.823555,0.004215,1.823555,0.004215,0.170950,0.205330,0.075485,...,4691.167609,2.106399e+07,1.887352e+07,1.445880e+07,8.410221e+06,3.456752e+06,2.051331e+07,628651.747865,2.192682e+07,2.200705e+07
2,7.5,0.118909,0.076977,2.138230,0.064388,2.138230,0.064388,0.467054,0.455603,0.084232,...,5240.483300,2.565806e+07,2.153269e+07,1.582927e+07,9.604824e+06,4.309011e+06,2.258887e+07,726072.939354,2.633922e+07,2.746267e+07
3,5.0,0.171120,0.021324,1.792696,0.052578,1.792696,0.052578,0.325038,0.320481,0.027322,...,5264.861124,2.444310e+07,1.840185e+07,1.128179e+07,5.461000e+06,2.243029e+06,2.530585e+07,440550.238293,2.770277e+07,2.771876e+07
4,7.5,0.173489,0.102469,1.675362,0.123744,1.675362,0.123744,0.288844,0.234557,0.076942,...,5855.434357,2.945279e+07,2.191223e+07,1.381792e+07,7.508275e+06,3.380258e+06,3.343349e+07,578449.096509,3.536981e+07,3.428611e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9982,2.5,0.146501,0.211585,1.582692,0.030286,1.582692,0.030286,0.251689,0.019034,0.118196,...,3575.027604,1.090331e+07,9.161639e+06,7.545964e+06,5.360013e+06,2.352602e+06,1.721655e+07,473781.026832,1.488078e+07,1.278082e+07
9983,2.5,0.126571,0.155664,1.813134,0.073643,1.813134,0.073643,0.368325,0.082026,0.283316,...,3779.664226,1.267819e+07,1.044614e+07,7.978084e+06,5.308375e+06,2.292214e+06,1.632097e+07,444454.488074,1.538019e+07,1.428586e+07
9984,2.5,0.027038,0.187394,1.344542,0.062589,1.344542,0.062589,0.112936,0.021800,0.168997,...,3460.810077,1.114673e+07,9.837551e+06,8.186796e+06,5.869743e+06,2.673834e+06,1.244380e+07,606753.617560,1.235763e+07,1.197721e+07
9985,2.5,0.010584,0.170055,1.266306,0.063249,1.266306,0.063249,0.048251,0.033863,0.009982,...,3941.466923,1.360459e+07,1.112793e+07,8.915723e+06,6.415134e+06,2.781860e+06,1.665226e+07,662080.766448,1.643196e+07,1.553516e+07
